# Глава 3 — мастер-дашборд (все 4 детектора)

Единая панель наблюдения за обучением. Показывает прогресс четырёх детекторов одновременно: YOLOv12 / RT-DETR / Faster R-CNN / DETR.

Для детальных графиков одного детектора — откройте соответствующий `chapter3_<detector>.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time
import subprocess
from IPython.display import clear_output, display, HTML

ROOT = Path('/home/vanusha/diplom/diploma-plant-disease')
DETECTORS = {
    'yolov12': {'task': 'task_07', 'log': '/tmp/train_yolov12.log', 'ultralytics': True},
    'rtdetr': {'task': 'task_08', 'log': '/tmp/train_rtdetr.log', 'ultralytics': True},
    'faster_rcnn': {'task': 'task_09', 'log': '/tmp/train_faster_rcnn.log', 'ultralytics': False},
    'detr': {'task': 'task_10', 'log': '/tmp/train_detr.log', 'ultralytics': False},
}
VARIANTS = ['baseline', 'aug_geom', 'aug_oversample', 'aug_diffusion']

## 🔴 Живой дашборд (auto-refresh 15s)

In [ ]:
def last_log_line(log_path):
    p = Path(log_path)
    if not p.exists():
        return '— не запущено —'
    lines = [l for l in p.read_text(errors='ignore').splitlines() if l.strip()]
    return lines[-1][:180] if lines else '(пусто)'

def gpu_status():
    try:
        out = subprocess.check_output(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu', '--format=csv,noheader'], timeout=5)
        return out.decode().strip()
    except Exception:
        return 'nvidia-smi недоступен'

def load_df(det_name, variant):
    det = DETECTORS[det_name]
    if det['ultralytics']:
        p = ROOT / 'code/results' / det['task'] / f'{det_name}_{variant}' / 'results.csv'
        if not p.exists():
            return None
        df = pd.read_csv(p); df.columns = [c.strip() for c in df.columns]
        if 'metrics/mAP50(B)' in df.columns:
            df = df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].copy()
            df.columns = ['epoch', 'val_mAP50', 'val_mAP5095']
            return df
        return None
    else:
        p = ROOT / 'code/results' / det['task'] / f'{det_name}_{variant}' / 'metrics.csv'
        if not p.exists():
            return None
        return pd.read_csv(p)

try:
    while True:
        clear_output(wait=True)
        now = time.strftime('%H:%M:%S')
        print(f'[dashboard {now}]  GPU: {gpu_status()}')
        print()
        for d in DETECTORS:
            print(f'  {d:14s} : {last_log_line(DETECTORS[d]["log"])}')

        # 4 subplots: one per detector, mAP@50 by variant
        fig, axs = plt.subplots(2, 2, figsize=(14, 9))
        for ax, d in zip(axs.flatten(), DETECTORS):
            any_data = False
            for v in VARIANTS:
                df = load_df(d, v)
                if df is None or df.empty:
                    continue
                ax.plot(df['epoch'], df['val_mAP50'], label=v, marker='o', markersize=2)
                any_data = True
            ax.set_title(f'{d}  —  val mAP@50')
            ax.set_xlabel('epoch'); ax.set_ylabel('mAP@50')
            ax.grid(True, alpha=0.3)
            if any_data:
                ax.legend(fontsize=8)
            else:
                ax.text(0.5, 0.5, '(ещё нет данных)', ha='center', va='center', transform=ax.transAxes, alpha=0.5)
        plt.tight_layout(); plt.show()

        # Summary table
        rows = []
        for d in DETECTORS:
            sp = ROOT / 'code/results' / DETECTORS[d]['task'] / 'summary.csv'
            if sp.exists():
                dfs = pd.read_csv(sp)
                for _, r in dfs.iterrows():
                    rows.append({'detector': d, **r.to_dict()})
        if rows:
            print('\n=== Сводка (на текущий момент) ===')
            display(pd.DataFrame(rows))

        time.sleep(15)
except KeyboardInterrupt:
    print('\n[dashboard stopped]')